In [9]:
import pandas as pd

# Load original dataset
df = pd.read_csv("full_data_unhealthy_imputed.csv")

# Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# Convert date column (ISO format auto-detects correctly)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Check how many NaT (if any)
print("NaT count in date:", df['date'].isna().sum())

# Create 8-hour bins (0, 8, 16)
df['hour_bin'] = (df['hour'] // 8) * 8

# Columns to aggregate
numeric_cols = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL']
target_cols = ['oestrus','calving','lameness','mastitis','disturbance']

# Grouping keys
group_cols = ['cow', 'date', 'hour_bin']

# Aggregation rules
agg_dict = {col: 'mean' for col in numeric_cols}
agg_dict.update({col: 'max' for col in target_cols})

# Perform the aggregation
df_reduced = df.groupby(group_cols).agg(agg_dict).reset_index()

# Show shapes
print("Original dataset shape:", df.shape)
print("Reduced dataset shape:", df_reduced.shape)

# Save reduced dataset
reduced_filename = "full_data_unhealthy_imputed_reduced.csv"
df_reduced.to_csv(reduced_filename, index=False)
print(f"Reduced dataset saved as {reduced_filename}")

NaT count in date: 0
Original dataset shape: (1935667, 16)
Reduced dataset shape: (318166, 12)
Reduced dataset saved as full_data_unhealthy_imputed_reduced.csv


In [11]:
import pandas as pd

# Load the reduced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced.csv")

# 1. Shape of dataset
print("Shape of dataset:", df.shape)

# 2. Column names
print("\nColumns in dataset:\n", df.columns.tolist())

# 3. Data types
print("\nData types:\n", df.dtypes)

# 4. Preview the first 5 rows
print("\nFirst 5 rows:\n", df.head())

# 5. Count of disease cases (1's) for each label
disease_cols = ['lameness', 'mastitis', 'calving', 'oestrus']
print("\nCount of disease presence (1) per column:")
for col in disease_cols:
    print(f"{col}: {df[col].sum()} out of {len(df)} rows")

# 6. Percentage of disease cases
print("\nPercentage of rows with disease present:")
for col in disease_cols:
    percent = (df[col].sum() / len(df)) * 100
    print(f"{col}: {percent:.4f}%")

# 7. Basic statistics for numeric columns
print("\nNumeric summary statistics:\n", df.describe())


Shape of dataset: (318166, 12)

Columns in dataset:
 ['cow', 'date', 'hour_bin', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'oestrus', 'calving', 'lameness', 'mastitis', 'disturbance']

Data types:
 cow                 int64
date               object
hour_bin            int64
IN_ALLEYS         float64
REST              float64
EAT               float64
ACTIVITY_LEVEL    float64
oestrus             int64
calving             int64
lameness            int64
mastitis            int64
disturbance         int64
dtype: object

First 5 rows:
    cow        date  hour_bin   IN_ALLEYS         REST          EAT  \
0  156  2015-03-02         0  261.545143  2635.472143   702.981714   
1  156  2015-03-02         8  576.327000   989.101125  2010.254625   
2  156  2015-03-02        16  194.088250  2065.915000  1335.476125   
3  156  2015-03-02        24    0.000000  3599.999000     0.000000   
4  156  2015-03-03         0  348.276857  2644.543000   599.872571   

   ACTIVITY_LEVEL  oestrus  calving

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load reduced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced.csv")

# Drop non-feature columns
drop_cols = ['cow', 'date']  # keep hour_bin
target_cols = ['lameness', 'mastitis', 'calving', 'oestrus']

X = df.drop(columns=drop_cols + target_cols)
y = df[target_cols]

print("Features used for modeling:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


Features used for modeling: ['hour_bin', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'disturbance']
X shape: (318166, 6)
y shape: (318166, 4)


In [15]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Split once (we will reuse the same split for all targets)
X_train, X_test, y_train_all, y_test_all = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

models = {}
for target in target_cols:
    print(f"\n=== Training model for {target} ===")

    # Select y for the target
    y_train = y_train_all[target]
    y_test = y_test_all[target]

    # Apply SMOTE only on training data
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    # Calculate scale_pos_weight for imbalance
    pos_weight = (len(y_train_res) - sum(y_train_res)) / sum(y_train_res)

    # Initialize XGBoost classifier
    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

    # Train model
    model.fit(X_train_res, y_train_res)

    # Evaluate on test set
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, digits=4))

    models[target] = model



=== Training model for lameness ===
              precision    recall  f1-score   support

           0     0.9987    0.8816    0.9365     63519
           1     0.0058    0.3826    0.0115       115

    accuracy                         0.8807     63634
   macro avg     0.5023    0.6321    0.4740     63634
weighted avg     0.9969    0.8807    0.9348     63634


=== Training model for mastitis ===
              precision    recall  f1-score   support

           0     0.9994    0.8707    0.9306     63593
           1     0.0007    0.1463    0.0015        41

    accuracy                         0.8702     63634
   macro avg     0.5000    0.5085    0.4660     63634
weighted avg     0.9987    0.8702    0.9300     63634


=== Training model for calving ===
              precision    recall  f1-score   support

           0     0.9990    0.9198    0.9578     63497
           1     0.0156    0.5912    0.0305       137

    accuracy                         0.9191     63634
   macro avg     0

In [17]:
import numpy as np
import pandas as pd

# Load reduced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced.csv")

# Sort by cow, then date, then hour_bin
df = df.sort_values(by=['cow','date','hour_bin'])

# 1. Cyclical time features
df['hour_sin'] = np.sin(2 * np.pi * df['hour_bin'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_bin'] / 24)

# 2. Rolling features
rolling_cols = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL','disturbance']
for col in rolling_cols:
    df[f'{col}_rollmean3'] = (
        df.groupby('cow')[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )
    df[f'{col}_rollstd3'] = (
        df.groupby('cow')[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).std())
        .fillna(0)
    )

# 3. Behavior ratios
total = df['EAT'] + df['REST'] + df['IN_ALLEYS']
df['EAT_ratio'] = df['EAT'] / total.replace(0, np.nan)
df['REST_ratio'] = df['REST'] / total.replace(0, np.nan)
df = df.fillna(0)

# Save enhanced dataset
enhanced_filename = "full_data_unhealthy_imputed_reduced_enhanced.csv"
df.to_csv(enhanced_filename, index=False)

print("Feature engineering complete.")
print("New shape:", df.shape)
print("Sample columns:", df.columns.tolist())


Feature engineering complete.
New shape: (318166, 26)
Sample columns: ['cow', 'date', 'hour_bin', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'oestrus', 'calving', 'lameness', 'mastitis', 'disturbance', 'hour_sin', 'hour_cos', 'IN_ALLEYS_rollmean3', 'IN_ALLEYS_rollstd3', 'REST_rollmean3', 'REST_rollstd3', 'EAT_rollmean3', 'EAT_rollstd3', 'ACTIVITY_LEVEL_rollmean3', 'ACTIVITY_LEVEL_rollstd3', 'disturbance_rollmean3', 'disturbance_rollstd3', 'EAT_ratio', 'REST_ratio']


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Load enhanced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# Drop non-feature columns
drop_cols = ['cow', 'date']
target_cols = ['lameness', 'mastitis', 'calving', 'oestrus']

X = df.drop(columns=drop_cols + target_cols)
y = df[target_cols]

print("Enhanced feature count:", X.shape[1])

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train_all, y_test_all = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train a model for each target
models = {}
for target in target_cols:
    print(f"\n=== Training model for {target} with enhanced features ===")

    y_train = y_train_all[target]
    y_test = y_test_all[target]

    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    pos_weight = (len(y_train_res) - sum(y_train_res)) / sum(y_train_res)

    model = XGBClassifier(
        n_estimators=400,
        max_depth=10,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_res, y_train_res)

    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, digits=4))

    models[target] = model


Enhanced feature count: 20

=== Training model for lameness with enhanced features ===
              precision    recall  f1-score   support

           0     0.9985    0.9831    0.9907     63519
           1     0.0174    0.1652    0.0315       115

    accuracy                         0.9817     63634
   macro avg     0.5079    0.5742    0.5111     63634
weighted avg     0.9967    0.9817    0.9890     63634


=== Training model for mastitis with enhanced features ===
              precision    recall  f1-score   support

           0     0.9994    0.9933    0.9963     63593
           1     0.0000    0.0000    0.0000        41

    accuracy                         0.9927     63634
   macro avg     0.4997    0.4967    0.4982     63634
weighted avg     0.9987    0.9927    0.9957     63634


=== Training model for calving with enhanced features ===
              precision    recall  f1-score   support

           0     0.9990    0.9921    0.9956     63497
           1     0.1328    0.56

In [21]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, make_scorer, f1_score

# Choose which target to tune first (do this for each)
target = 'calving'  # You can change to lameness, mastitis, oestrus

# Extract X_train, y_train, etc.
y_train = y_train_all[target]
y_test = y_test_all[target]

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Dynamic scale_pos_weight
pos_weight = (len(y_train_res) - sum(y_train_res)) / sum(y_train_res)

# Base model
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=pos_weight
)

# Hyperparameter grid
param_dist = {
    'n_estimators': np.arange(200, 601, 100),
    'max_depth': np.arange(5, 13),
    'learning_rate': np.linspace(0.01, 0.2, 10),
    'subsample': np.linspace(0.7, 1.0, 4),
    'colsample_bytree': np.linspace(0.7, 1.0, 4),
    'gamma': np.linspace(0, 5, 6),
    'min_child_weight': np.arange(1, 11)
}

# Custom scorer: f1-score (focus on minority class)
scorer = make_scorer(f1_score, average='binary')

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,  # can increase for better search
    scoring=scorer,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit search
random_search.fit(X_train_res, y_train_res)

# Best params
print("Best params for", target, ":", random_search.best_params_)

# Evaluate on test set
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nClassification report on test set:")
print(classification_report(y_test, y_pred, digits=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best params for calving : {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 4, 'max_depth': 12, 'learning_rate': 0.1788888888888889, 'gamma': 0.0, 'colsample_bytree': 0.7999999999999999}

Classification report on test set:
              precision    recall  f1-score   support

           0     0.9989    0.9940    0.9964     63497
           1     0.1508    0.4964    0.2313       137

    accuracy                         0.9929     63634
   macro avg     0.5748    0.7452    0.6139     63634
weighted avg     0.9971    0.9929    0.9948     63634

